# 04 · Retrieve — 01 Single-Index Retrieval (the simple path)

**No API key and no network access are required to run this notebook end to end — it runs entirely on a local, in-memory per-course store.**

Implements `retrieve()`, `upsert_chunks()`, and `max_score()` for a single
per-course/per-tenant index. In production this queries a Pinecone
namespace per course, falling back to a local JSON store when no vector
database is configured. The Pinecone leg is not exercised here (no key, no
index) — this notebook runs the **local fallback path**, which is the same
code shape with a plain dict standing in for the JSON-on-disk store.

**In → out:** a course id + a query string → that course's own chunks, ranked
by cosine similarity, plus a `max_score` / `is_grounded` readout.

**What this notebook exists to demonstrate:** course isolation. A query issued
against course A structurally cannot return course B's material — not because
it scores lower, but because retrieval for course A never reads anything
outside course A's own store. Two small synthetic "courses" are built inline
below (invented for this notebook — no real course content) to make that
concrete.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `embed_text` | Offline, deterministic hash embedding — no model, no network. | `embed_text("mitochondria...", dim=64)` → a 64-dim unit vector |
| `_cosine` | Cosine similarity between two vectors, with a dimension-mismatch guard. | `_cosine(qvec, chunk_vec)` → a float in `[-1, 1]` |
| `upsert_chunks` | Embeds and stores chunks for one course id. | `upsert_chunks("bio201", bio201_chunks)` → `3` |
| `_score_chunks` | Scores a course's stored chunks against a query vector, sorted descending. | `_score_chunks(chunks, qvec, "bio201")` → scored list |
| `retrieve` | Top-k chunks scoped to one course's own store — never another course's. | `retrieve("bio201", "what do mitochondria do?", k=3)` |
| `max_score` / `is_grounded` | The guardrail: is the top hit even good enough to trust? | `is_grounded(bio_hits)` → `True`/`False` |


In [ ]:
# Locate the repo root before `import nbio` can work at all -- this notebook lives
# two directories below it (01-modules/04-retrieve/), and both a plain `jupyter
# nbconvert --execute` and some Jupyter front ends start the kernel with this
# notebook's own directory as cwd, not the repo root.
import sys
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(6):
    if (_p / "nbio.py").is_file():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")

import nbio

repo_root = nbio.bootstrap()
nbio.show_environment()

## Step 1 — the offline embedding fallback

Same deterministic, dependency-free hash embedding used as the offline path
in stage `03-embed`: a bag-of-hashed-words vector, L2-normalized. It is real
wiring, not a mock — but the vectors are **not semantically meaningful**, only
consistent, so cosine similarity between two texts that share words is higher
than between two that don't. That's all `course_rag.py`'s local fallback
needs to demonstrate the property this notebook is about.

In [ ]:
import hashlib
import math

EMBED_DIM = 64


def embed_text(text: str, dim: int = EMBED_DIM) -> list[float]:
    vec = [0.0] * dim
    for tok in (text or "").lower().split():
        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)
        vec[h % dim] += 1.0
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]

## Step 2 — cosine similarity, with a dimension-mismatch guard

Two vectors of different length can't be compared, so this returns `0.0`
instead of raising — the same defensive choice made for a mixed-dimension
index elsewhere (see stage `03-embed`'s `03-dimensions.ipynb`).

In [ ]:
def _cosine(a: list[float], b: list[float]) -> float:
    if not a or not b or len(a) != len(b):
        return 0.0
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a)) or 1.0
    nb = math.sqrt(sum(x * x for x in b)) or 1.0
    return dot / (na * nb)

## Step 3 — two synthetic "courses" (anchor data)

Invented for this notebook only — no real course, student, or partner
content. `bio201` is a short cell-biology course; `hist101` is a short
world-history course. Nothing links them. Defined here, before the store
logic below, so the store and retrieval functions have real data to act on
from their first call.

In [ ]:
bio201_chunks = [
    {
        "chunk_id": "bio201-1",
        "text": "Mitochondria are the primary site of ATP production in a eukaryotic cell, "
                "generating energy through oxidative phosphorylation across the inner membrane.",
        "source_file": "cell_biology_notes.md",
        "module": "organelles",
    },
    {
        "chunk_id": "bio201-2",
        "text": "The nucleus stores the cell's genetic material and controls gene expression "
                "through transcription, separated from the cytoplasm by the nuclear envelope.",
        "source_file": "cell_biology_notes.md",
        "module": "organelles",
    },
    {
        "chunk_id": "bio201-3",
        "text": "Ribosomes translate messenger RNA into polypeptide chains, either free in the "
                "cytosol or bound to the rough endoplasmic reticulum.",
        "source_file": "cell_biology_notes.md",
        "module": "protein-synthesis",
    },
]

hist101_chunks = [
    {
        "chunk_id": "hist101-1",
        "text": "The Congress of Vienna in 1815 redrew the map of Europe after the Napoleonic "
                "Wars, restoring several monarchies and establishing a new balance of power.",
        "source_file": "world_history_notes.md",
        "module": "19th-century-europe",
    },
    {
        "chunk_id": "hist101-2",
        "text": "The Silk Road was a network of trade routes connecting East Asia to the "
                "Mediterranean, carrying silk, spices, and ideas across Eurasia for centuries.",
        "source_file": "world_history_notes.md",
        "module": "trade-routes",
    },
    {
        "chunk_id": "hist101-3",
        "text": "The printing press, developed by Gutenberg around 1440, dramatically lowered "
                "the cost of reproducing text and accelerated the spread of literacy in Europe.",
        "source_file": "world_history_notes.md",
        "module": "technology",
    },
]

print(f"bio201: {len(bio201_chunks)} chunks   hist101: {len(hist101_chunks)} chunks")

## Step 4 — store chunks for one course

In production this store is keyed by `(tenant_id, course_id)` and persisted
to `courses/{course_id}/rag_store/{tenant}_chunks.json` on disk, resolved
through a settings/course-loader layer. None of that multi-tenant plumbing
is needed here — the shape that matters is a chunk store **keyed by course
id**, and `retrieve(course_id, ...)` (Step 6) only ever reads
`store[course_id]`. That's the whole mechanism behind course isolation;
everything else — Pinecone/pgvector production backends (same interface,
different storage), or settings plumbing — is orthogonal to the point this
notebook makes.

In [ ]:
from typing import Any

_STORE: dict[str, list[dict[str, Any]]] = {}


def upsert_chunks(course_id: str, chunks: list[dict[str, Any]]) -> int:
    '''Embed and store chunks for one course. Returns count upserted.'''
    if not chunks:
        return 0
    enriched: list[dict[str, Any]] = []
    for ch in chunks:
        text = str(ch.get("text") or "").strip()
        if not text:
            continue
        enriched.append({**ch, "course_id": course_id, "embedding": embed_text(text)})

    existing = _STORE.get(course_id, [])
    by_id = {str(c.get("chunk_id")): c for c in existing if c.get("chunk_id")}
    for ch in enriched:
        by_id[str(ch["chunk_id"])] = ch
    _STORE[course_id] = list(by_id.values())
    return len(enriched)

In [ ]:
n1 = upsert_chunks("bio201", bio201_chunks)
n2 = upsert_chunks("hist101", hist101_chunks)
print(f"upserted {n1} chunks into bio201, {n2} chunks into hist101")  # <- look: how many stored?

## Step 5 — score a query against that course's chunks

`_score_chunks` cosine-scores every chunk already stored for one course
against a query vector and sorts descending — no cross-course reads, since
it only ever receives the chunks `retrieve` (Step 6) hands it from that one
course's own store.

In [ ]:
def _score_chunks(chunks: list[dict[str, Any]], qvec: list[float], course_id: str) -> list[dict[str, Any]]:
    scored = []
    for ch in chunks:
        vec = ch.get("embedding")
        if not isinstance(vec, list):
            continue
        scored.append({**ch, "score": _cosine(qvec, vec), "course_id": course_id})
    scored.sort(key=lambda x: -float(x.get("score") or 0))
    return scored

## Step 6 — the `retrieve` function contributors will actually reuse

`retrieve(course_id, ...)` reads `_STORE.get(course_id, [])` and nothing
else — that single line is the entire isolation guarantee.

In [ ]:
def retrieve(course_id: str, query: str, k: int = 5) -> list[dict[str, Any]]:
    '''Top-k chunks scoped to course_id's own store -- never another course's.'''
    cid = (course_id or "").strip()
    q = (query or "").strip()
    if not cid or not q:
        return []
    chunks = _STORE.get(cid, [])
    if not chunks:
        return []
    qvec = embed_text(q)
    return _score_chunks(chunks, qvec, cid)[:k]

In [ ]:
bio_query = "What is the function of mitochondria in a cell?"

bio_hits = retrieve("bio201", bio_query, k=3)
nbio.table(
    [(h["chunk_id"], f"{h['score']:.3f}", h["text"][:70]) for h in bio_hits],
    headers=("chunk_id", "score", "text"),
)

## Step 7 — is the top hit good enough to trust?

`max_score` / `is_grounded` are the guardrail on top of `retrieve`: a
neutral, illustrative threshold (`GROUNDING_THRESHOLD = 0.15`, not a product
config value) below which the top hit is treated as "not grounded" rather
than presented as an answer.

In [ ]:
def max_score(chunks: list[dict[str, Any]]) -> float:
    if not chunks:
        return 0.0
    return max(float(c.get("score") or 0) for c in chunks)


# A neutral example threshold -- not a product config value.
GROUNDING_THRESHOLD = 0.15


def is_grounded(chunks: list[dict[str, Any]]) -> bool:
    return bool(chunks) and max_score(chunks) >= GROUNDING_THRESHOLD

In [ ]:
print(f"max_score={max_score(bio_hits):.3f}  is_grounded={is_grounded(bio_hits)}")

## Step 8 — same query against `hist101` — course isolation

This is the property the notebook is built to show. `hist101`'s store never
contained anything about mitochondria -- so the *same* biology query, run
against the history course, cannot return biology material at all. It can
only ever rank `hist101`'s own three chunks against a query they have almost
nothing to do with.

Note the score below is not necessarily lower than the in-domain query above
-- the hash embedding is deterministic, not semantically meaningful (same
caveat as stage `03-embed`'s offline path), so a short, generic-vocabulary
query can hash into overlapping buckets by coincidence. That's the honest
limit of this fallback, and it's exactly why the next cell checks the
guarantee that actually matters -- which chunks are even in the running --
rather than trusting the score alone.

In [ ]:
hist_hits_for_bio_query = retrieve("hist101", bio_query, k=3)
nbio.table(
    [(h["chunk_id"], f"{h['score']:.3f}", h["text"][:70]) for h in hist_hits_for_bio_query],
    headers=("chunk_id", "score", "text"),
)

In [ ]:
print(f"max_score={max_score(hist_hits_for_bio_query):.3f}  "
      f"is_grounded={is_grounded(hist_hits_for_bio_query)}")

## Step 9 — proving it structurally, not just by score

A lower score isn't the guarantee -- with a hash-based fallback embedding, a
short cross-domain query can coincidentally score *higher* than an in-domain
one, as it may have just done above. The actual guarantee -- the one that
holds regardless of embedding quality -- is that `hist101`'s result set is
always a subset of `hist101`'s own chunk ids, full stop; `bio201`'s chunk ids
are never even candidates, because `retrieve("hist101", ...)` never reads
`_STORE["bio201"]` in the first place. That's checked directly below.

In [ ]:
hist_ids = {c["chunk_id"] for c in hist101_chunks}
bio_ids = {c["chunk_id"] for c in bio201_chunks}
returned_ids = {h["chunk_id"] for h in hist_hits_for_bio_query}

assert returned_ids <= hist_ids, "hist101 query returned a chunk not in hist101's own store"
assert not (returned_ids & bio_ids), "hist101 query returned a bio201 chunk -- isolation broken"

print("returned chunk ids  :", sorted(returned_ids))
print("subset of hist101's own store:", returned_ids <= hist_ids)
print("disjoint from bio201's store :", not (returned_ids & bio_ids))

## Wrap-up

`retrieve`, `upsert_chunks`, and `max_score` keep the same three-function
shape the product uses across all three backends (Pinecone, pgvector, local
JSON) -- only the storage call inside changes. This notebook exercised the
local fallback exactly as `course_rag.py` does whenever no Pinecone key is
configured, which is also the path a contributor gets by default with no
setup at all.

Next: `02-multi-source-fanout.ipynb` -- the other implementation, which
trades this single-index simplicity for six sources and a five-signal
ranking blend.